# 🏛️ LE SYNDICAT (Monster + Sniper + Sénat)

## La Stratégie Finale (OPTIMAL 0.15)
Ce notebook implémente l'architecture hybride ultime :
1.  **Base Solide** : On prend l'excellent **FN_SNIPER** (F1 0.764) comme fondation. (On ne jette rien).
2.  **Apport du Sénat** : On regarde ce que le **Sénat V9** a trouvé en plus (Recall).
3.  **Filtre du Monstre** : On utilise le **Pseudo-Labeling Monster** pour valider ces ajouts du Sénat.
    *   Si le Monstre donne une probabilité > **0.15**, on accepte l'ajout (Bonus).
    *   Sinon, on rejette (Bruit).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

## 1. Chargement & Monster Training
On ré-entraîne le Monster pour avoir ses probabilités fraîches.

In [ ]:
print("📥 Chargement des datasets...")
train_df = pd.read_csv('conversion_data_train.csv')
test_df = pd.read_csv('conversion_data_test.csv')
v9_labels = pd.read_csv('submission_LE_SENAT_V9.csv')

# Pseudo-Labels V9 pour le Test
test_df['converted'] = v9_labels['converted']

# Monster Dataset (Train + Test)
full_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

# Feature Engineering
def feature_engineering(df):
    df = df.copy()
    df['pages_per_age'] = df['total_pages_visited'] / (df['age'] + 0.1)
    df['interaction_age_pages'] = df['age'] * df['total_pages_visited']
    df['is_active'] = (df['total_pages_visited'] > 2).astype(int)
    return df

X_full = feature_engineering(full_df.drop('converted', axis=1))
y_full = full_df['converted']
X_test = feature_engineering(test_df.drop('converted', axis=1))

# Encoding
cat_cols = ['country', 'source']
cat_indices = []
for col in cat_cols:
    le = LabelEncoder()
    le.fit(X_full[col])
    X_full[col] = le.transform(X_full[col])
    X_test[col] = le.transform(X_test[col])
    cat_indices.append(X_full.columns.get_loc(col))

# Training
print("🔥 Entraînement du Monster (HistGradientBoosting)...")
model = HistGradientBoostingClassifier(
    loss="log_loss", learning_rate=0.03, max_iter=500, max_depth=8,
    l2_regularization=0.1, categorical_features=cat_indices, random_state=42
)
model.fit(X_full, y_full)
print("✅ Modèle entraîné.")

## 2. La Logique du Syndicat (Optimal)
On combine les forces des 3 modèles.

In [ ]:
# 1. Probabilités du Monstre (La jauge de confiance)
final_proba = model.predict_proba(X_test)[:, 1]

# 2. Chargement du Champion Actuel (Sniper)
sniper_df = pd.read_csv('submission_FN_SNIPER.csv')
sniper_pred = sniper_df['converted']

# 3. Formule Magique (Threshold 0.15)
# On garde TOUT le Sniper (Safe).
# On ajoute le Sénat SI le Monstre n'est pas totalement contre (> 0.15)
monster_yes = (final_proba >= 0.5).astype(int)
senate_yes = v9_labels['converted']
safety_check = (final_proba >= 0.15).astype(int)

syndicate_pred = (monster_yes | sniper_pred | (senate_yes & safety_check)).astype(int)

# 4. Export
submission = pd.DataFrame({'converted': syndicate_pred})
filename = 'submission_SYNDICATE_OPTIMAL.csv'
submission.to_csv(filename, index=False)

# 5. Analyse des Gains
sniper_count = sniper_pred.sum()
final_count = syndicate_pred.sum()
gain = final_count - sniper_count

print(f"✅ Fichier généré : {filename}")
print(f"📊 Bilan Final :")
print(f"   - Sniper (Base) : {sniper_count} conversions")
print(f"   - Syndicat      : {final_count} conversions")
print(f"   -> Gain Net     : +{gain} conversions (Bonus Sénat filtré)")
print("🚀 Prêt à soumettre !")